# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

# 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World


Free GPU Memory (GB): 44.5264


In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Tue_May__3_18:49:52_PDT_2022
Cuda compilation tools, release 11.7, V11.7.64
Build cuda_11.7.r11.7/compiler.31294372_0


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 348.01 GB
MemAvailable: 972.70 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successfu

# 2. Apply perturbations

In [6]:
# Import all functions from the provided code
from typing import List, Dict
import copy

from src.reliability.apply_typos import apply_typo_modifications

# First define the perturbation types that don't require external resources
PERTURBATION_TYPES = [
    'char_insert_noise',
    'char_substitution', 
    'char_insertion',
    'char_deletion',
    'char_replacement',
    'char_repetition',
    'char_swapping',
    'char_LCC',
    'word_emoji',
    'word_internet_slang',
    'word_phrase_translation',
    'word_repeat',
    'word_context_aware_insertion',
    'word_remove_punctuation',
    'word_keyword_only',
    'word_CMW',
    'word_synonym'
]

def apply_single_perturbation(query: str, perturbation_type: str, intensity: int = 3) -> str:
    """Apply a single perturbation type with given intensity."""
    # Create typo dictionary with only one perturbation type
    typo_dict = {p: 0 for p in PERTURBATION_TYPES}
    typo_dict[perturbation_type] = intensity
    
    # Apply the perturbation
    return apply_typo_modifications(query, typo_dict, [])

def main():
    original_query = "What is the capital of France?"
    intensity = 3
    
    print(f"Original query: {original_query}\n")
    print("Applying perturbations with intensity {intensity}:\n")
    
    # Apply each perturbation type sequentially
    for perturbation_type in PERTURBATION_TYPES:
        try:
            perturbed_query = apply_single_perturbation(original_query, perturbation_type, intensity)
            print(f"Perturbation type: {perturbation_type}")
            print(f"Perturbed query: {perturbed_query}")
            print("-" * 80 + "\n")
        except Exception as e:
            print(f"Error applying {perturbation_type}: {str(e)}")
            print("-" * 80 + "\n")

if __name__ == "__main__":
    main()

Original query: What is the capital of France?

Applying perturbations with intensity {intensity}:

Perturbation type: char_insert_noise
Perturbed query: What is' the \capital of4 France?
--------------------------------------------------------------------------------

Perturbation type: char_substitution
Perturbed query: What is t#e cap1tal of F®ance?
--------------------------------------------------------------------------------

Perturbation type: char_insertion
Perturbed query: Wnhat is the capital ofK FraHnce?
--------------------------------------------------------------------------------

Perturbation type: char_deletion
Perturbed query: What s the capial o France?
--------------------------------------------------------------------------------

Perturbation type: char_replacement
Perturbed query: Wyat is the cwpital od France?
--------------------------------------------------------------------------------

Perturbation type: char_repetition
Perturbed query: Whhat iss the capi